# AIML Capstone — Autonomous Driving
**Business scenario:** Autonomous vehicles (AV) and intelligent transport systems (ITS) are the future of road transport. This notebook covers both parts of the capstone:

- **Part 1 — Object Detection:** a deep-learning model that predicts the vehicle type present in an image and localizes it with a rectangular bounding box.
- **Part 2 — Data Science:** an exploratory analysis of the usage of Tesla Autopilot and its effect on road safety.

This notebook is the source-code companion to the deployed Flask app (`app/`). The same `detector.py` and `analysis.py` modules that power the live web app are used here so results stay consistent between the notebook and the deployment.


## 0. Setup

In [ ]:
# If running outside the project environment, install dependencies first:
# !pip install opencv-python-headless numpy pandas matplotlib seaborn requests

import sys, os
sys.path.append(os.path.abspath("../app"))  # so we can import detector.py / analysis.py

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2

sns.set_style("whitegrid")
%matplotlib inline


---
## Part 1 — Object Detection

### 1.1 Dataset
The capstone dataset (`Images.zip`) contains traffic-camera style images of vehicles on the road. In this notebook we work with a `sample_images/` folder — point `IMAGE_DIR` at your unzipped dataset to run detection over the full set.

### 1.2 Model / architecture choice
Rather than training a CNN object detector from scratch (which needs a labelled bounding-box dataset, a GPU, and many epochs), this project uses **transfer learning**: a MobileNet-SSD convolutional network (pre-trained on Pascal VOC, which already includes `car`, `bus`, `motorbike`, `bicycle`, `train`) via OpenCV's DNN module.

This choice is deliberate for the deployment target (a free Render web service with limited RAM/CPU and no GPU):
- MobileNet-SSD is small (~23MB) and runs in real time on CPU.
- It already localizes objects with a bounding box **and** classifies them — exactly the two objectives in the brief.
- It avoids pulling in PyTorch/TensorFlow (multi-GB, slow cold starts on free hosting tiers).

If you have a GPU and want to fine-tune on the labelled capstone dataset instead, see the "Optional: fine-tuning a custom CNN" section at the end of Part 1.


In [ ]:
from detector import detect_vehicles, load_model, VEHICLE_CLASSES

# Loads the network, downloading weights on first run (needs internet)
model = load_model()
print("Model loaded. Vehicle classes recognized:", VEHICLE_CLASSES)


### 1.3 Run inference on sample images

In [ ]:
IMAGE_DIR = "../data/sample_images"  # point this at your unzipped Images.zip folder
os.makedirs(IMAGE_DIR, exist_ok=True)

image_files = [f for f in os.listdir(IMAGE_DIR) if f.lower().endswith((".jpg", ".jpeg", ".png"))]
print(f"Found {len(image_files)} images in {IMAGE_DIR}")

results_summary = []

for fname in image_files[:20]:  # cap for a quick demo run
    path = os.path.join(IMAGE_DIR, fname)
    img = cv2.imread(path)
    if img is None:
        continue
    annotated, detections, inference_ms = detect_vehicles(img)
    results_summary.append({"file": fname, "n_vehicles": len(detections), "inference_ms": inference_ms})

    # Show inline
    plt.figure(figsize=(6, 4))
    plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
    plt.title(f"{fname} — {len(detections)} vehicle(s), {inference_ms:.1f} ms")
    plt.axis("off")
    plt.show()

pd.DataFrame(results_summary)


### 1.4 Evaluate detections
For a labelled dataset (ground-truth bounding boxes + classes), compute standard object-detection metrics:


In [ ]:
# Example scaffold for computing IoU / precision / recall against ground truth,
# once you have a labels file (e.g. COCO-style JSON or Pascal VOC XML) for the dataset.

def iou(box_a, box_b):
    xa1, ya1, xa2, ya2 = box_a
    xb1, yb1, xb2, yb2 = box_b
    inter_x1, inter_y1 = max(xa1, xb1), max(ya1, yb1)
    inter_x2, inter_y2 = min(xa2, xb2), min(ya2, yb2)
    inter_area = max(0, inter_x2 - inter_x1) * max(0, inter_y2 - inter_y1)
    area_a = (xa2 - xa1) * (ya2 - ya1)
    area_b = (xb2 - xb1) * (yb2 - yb1)
    union = area_a + area_b - inter_area
    return inter_area / union if union > 0 else 0

# ground_truths = load_your_labels(...)
# predictions   = [detect_vehicles(cv2.imread(p))[1] for p in image_paths]
# then match boxes with iou() >= 0.5 to compute precision/recall/mAP.
print("Plug in your labelled ground truth to compute IoU / precision / recall / mAP.")


### 1.5 Optional: fine-tuning a custom CNN (if you have GPU + labelled data)
If the assignment requires you to *train* your own architecture rather than use transfer learning, a minimal Keras/TensorFlow single-class detector skeleton looks like this (requires `tensorflow` — not installed in the lightweight deployment):


In [ ]:
# import tensorflow as tf
# from tensorflow.keras import layers, models
#
# def build_simple_detector(input_shape=(224, 224, 3), n_classes=5):
#     base = tf.keras.applications.MobileNetV2(input_shape=input_shape, include_top=False, weights="imagenet")
#     base.trainable = False
#     x = layers.GlobalAveragePooling2D()(base.output)
#     class_head = layers.Dense(n_classes, activation="softmax", name="class_output")(x)
#     box_head = layers.Dense(4, activation="sigmoid", name="box_output")(x)  # normalized [x1,y1,x2,y2]
#     return models.Model(base.input, [class_head, box_head])
#
# model = build_simple_detector()
# model.compile(
#     optimizer="adam",
#     loss={"class_output": "categorical_crossentropy", "box_output": "mse"},
# )
# model.fit(train_ds, validation_data=val_ds, epochs=20)
print("Skeleton only — wire up your labelled dataset to train.")


---
## Part 2 — Data Science: Autopilot & Road Safety

### 2.1 Load & inspect the data


In [ ]:
from analysis import load_data

df = load_data("../data/Tesla-Deaths.csv")
print(df.shape)
df.info()
df.head()


In [ ]:
# Preliminary inspection & cleaning
print("Duplicate rows:", df.duplicated().sum())
print("
Missing values per column:")
print(df.isna().sum())


### 2.2 Events over time (by date, year, weekday, state/country)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
df["Year"].value_counts().sort_index().plot(kind="bar", ax=axes[0], color="#3b82f6")
axes[0].set_title("Events per Year")

df["Country"].value_counts().plot(kind="bar", ax=axes[1], color="#f59e0b")
axes[1].set_title("Events per Country")
plt.tight_layout()
plt.show()

if "State" in df.columns:
    plt.figure(figsize=(10, 4))
    df[df["State"] != "-"]["State"].value_counts().plot(kind="bar", color="#10b981")
    plt.title("Events per US State")
    plt.show()

if "Weekday" in df.columns:
    order = ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]
    plt.figure(figsize=(8, 4))
    df["Weekday"].value_counts().reindex(order).plot(kind="bar", color="#8b5cf6")
    plt.title("Events per Day of Week")
    plt.show()


### 2.3 Answering the capstone's EDA questions

- What is the number of victims (deaths) in each accident?
- How many times did Tesla drivers die?
- What is the proportion of events in which one or more occupants died?
- What is the distribution of events in which the vehicle hit a cyclist or a pedestrian?
- How many times did the accident involve the death of an occupant/driver **and** a cyclist/pedestrian?
- What is the frequency of Tesla colliding with other vehicles?
- What is the event distribution across models?
- What is the distribution of verified Tesla Autopilot deaths?


In [ ]:
# Deaths per accident
plt.figure(figsize=(6,4))
df["Deaths"].value_counts().sort_index().plot(kind="bar", color="#ef4444")
plt.title("Number of Victims (Deaths) per Accident")
plt.xlabel("Deaths in a single accident"); plt.ylabel("Number of accidents")
plt.show()

tesla_driver_deaths = int(df["Tesla driver"].sum())
print(f"Tesla driver deaths: {tesla_driver_deaths}")

occupant_death_pct = round(100 * (df["Tesla driver"].add(df.get("Tesla occupant", 0)) > 0).mean(), 1)
print(f"Percent of events with >=1 occupant death: {occupant_death_pct}%")

cyclist_ped_events = int(df["Cyclists/Peds"].sum())
print(f"Events involving a cyclist/pedestrian: {cyclist_ped_events}")

both_events = int(((df["Tesla driver"].add(df.get("Tesla occupant", 0)) > 0) & (df["Cyclists/Peds"] > 0)).sum())
print(f"Events with BOTH an occupant death AND a cyclist/pedestrian: {both_events}")

other_vehicle_events = int(df["Other vehicle"].sum())
print(f"Events involving collision with another vehicle: {other_vehicle_events}")


In [ ]:
# Distribution across models
plt.figure(figsize=(6,4))
df["Model"].value_counts().plot(kind="bar", color="#8b5cf6")
plt.title("Event Distribution Across Tesla Models")
plt.show()

# Verified Autopilot deaths
plt.figure(figsize=(5,4))
df["Verified Tesla Autopilot Deaths"].value_counts().sort_index().plot(kind="bar", color="#0ea5e9")
plt.title("Verified Tesla Autopilot Deaths (per event)")
plt.show()

print("Total verified Autopilot-linked deaths:", int(df["Verified Tesla Autopilot Deaths"].sum()))


### 2.4 Summary of findings (fill in after running on the *real* dataset)

Replace `data/Tesla-Deaths.csv` with the actual dataset (same column schema) and re-run this
notebook top-to-bottom. Then summarize, e.g.:

- Trend in events per year (increasing / decreasing / flat, and possible reasons)
- Which states/countries account for the most events
- Share of events where Autopilot was verified as a contributing factor
- Whether cyclist/pedestrian involvement is rising alongside Autopilot adoption
- Any caveats about the data (self-reported, media-sourced, survivorship bias, etc.)
